## DSPy Prompts to Improve Rationales

### Setting up DSPy

In [ ]:
# Import libraries
import dspy
import json

import os
from dotenv import load_dotenv
load_dotenv()
open_ai_api_key = os.getenv("OPENAI_API_KEY")


c:\Users\Cassandra\miniconda3\envs\coalesce\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Set up the LM (https://dspy-docs.vercel.app/api/language_model_clients/OpenAI)
gpt3_turbo = dspy.OpenAI(model='gpt-3.5-turbo', max_tokens=500, api_key=open_ai_api_key)  
dspy.configure(lm=gpt3_turbo)

In [3]:
gpt3_turbo("which openai model are you? are you gpt-3.5-turbo?")

['I am an AI assistant powered by the GPT-3 model developed by OpenAI. I am not specifically the GPT-3.5-turbo model, but I am based on the GPT-3 architecture.']

In [4]:
test_inputs = [
    {
        "question": {
            "cell_type": "question",
            "response_format": "open",
            "description": "",
            "main_text": "How much do you trust your local government's decisions on public parks and how often do you visit these parks?",
            "response_categories": []
        },
        "problem": "lack of specificity",
        "input_rationale": "Let's think step by step in order to determine the specificity. We first need to identify if the question measures only one underlying concept. In this case, the question combines two concepts: trust in local government decisions regarding public parks and frequency of park visits. Therefore, the question is not specific in measuring only one underlying concept."
    },
    {
        "question": {
            "cell_type": "question",
            "response_format": "open",
            "description": "",
            "main_text": "How much do you trust your local government's decisions on public parks and how often do you visit these parks?",
            "response_categories": []
        },
        "problem": "lack of specificity",
        "input_rationale": "Let's think step by step in order to determine the specificity. We first need to identify if the question measures only one underlying concept."
    },
]

In [5]:
constants = {
    "question_json_format": """{
        "cell_type": "question",
        "response_format": "open" or "closed",
        "description": string,
        "main_text": string,
        "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions
    }"""
}

In [6]:
def parse_json_str(json_str):
    return json.loads(json_str)

def fill_in_constants(input_str):
    for key in constants:
        input_str = input_str.replace("{"+key+"}", constants[key])
    return input_str

### Creating Signature

In [7]:
sig_description = """Please take the inputted question, which has been flagged to have an inputted problem.
Read through the inputted rationale, which tries to explain why the question has a particular problem.
Please re-write the rationale in complete sentences to help users understand gaps in the question. 
Please keep your response between 20 and 50 words. 
Please start your response with "This question" followed by your rationale."""

input_descriptions = """{
    "question": "The question to evaluate. The input will be a JSON with the following structure: {question_json_format}",
    "problem": "The problem with the question. The input will be a string.",
    "input_rationale": "The rationale for the problem. The input will be a string."}"""

output_descriptions = """{"new_rationale": "The new rationale for the problem. The output will be a string."}"""

# convert the descriptions to JSON
input_descriptions_json = parse_json_str(input_descriptions)
output_descriptions_json = parse_json_str(output_descriptions)

class CleanRationale(dspy.Signature):

    question = dspy.InputField(desc=fill_in_constants(input_descriptions_json["question"]))
    problem = dspy.InputField(desc=input_descriptions_json["problem"])
    input_rationale = dspy.InputField(desc=input_descriptions_json["input_rationale"])
    new_rationale = dspy.OutputField(desc=output_descriptions_json["new_rationale"])

# set the signature description
CleanRationale.__doc__ = sig_description

print(CleanRationale.__doc__)

Please take the inputted question, which has been flagged to have an inputted problem.
Read through the inputted rationale, which tries to explain why the question has a particular problem.
Please re-write the rationale in complete sentences to help users understand gaps in the question. 
Please keep your response between 20 and 50 words. 
Please start your response with "This question" followed by your rationale.


In [8]:
# Create a module
class CleanRationaleModule(dspy.Module):
    def __init__(self):

        super().__init__()
        
        self.new_rationale = dspy.ChainOfThought(CleanRationale)

    def forward(self, question, input_rationale, problem, return_rationale=False, temp=0.7):

        output = self.new_rationale(question=question, 
                                    input_rationale=input_rationale, 
                                    problem=problem,
                                    config=dict(temperature=temp))

        # return the output as a dictionary

        if return_rationale:
            return {"new_rationale": output.new_rationale, "rationale": output.rationale}
        else:
            return {"new_rationale": output.new_rationale}

In [9]:
# Test CleanRationaleModule

clean_rationale_module = CleanRationaleModule()

for test_input in test_inputs:

    test_input_str = json.dumps(test_input["question"])

    print("Input:")
    print(test_input_str)
    print(test_input["input_rationale"])
    print(test_input["problem"])
    print("\n")

    output = clean_rationale_module(test_input_str, 
                                    test_input["input_rationale"], 
                                    test_input["problem"],
                                    return_rationale=True)
    print(output)
    print("\n")

Input:
{"cell_type": "question", "response_format": "open", "description": "", "main_text": "How much do you trust your local government's decisions on public parks and how often do you visit these parks?", "response_categories": []}
Let's think step by step in order to determine the specificity. We first need to identify if the question measures only one underlying concept. In this case, the question combines two concepts: trust in local government decisions regarding public parks and frequency of park visits. Therefore, the question is not specific in measuring only one underlying concept.
lack of specificity


{'new_rationale': 'The question lacks specificity by combining two concepts: trust in local government decisions and frequency of park visits. This dual focus prevents a clear measurement of a single underlying concept.', 'rationale': "This question lacks specificity as it combines two concepts, trust in local government decisions and frequency of park visits. By asking about 

In [10]:
# Let's look at the model history
gpt3_turbo.inspect_history(n=1)





Please take the inputted question, which has been flagged to have an inputted problem.
Read through the inputted rationale, which tries to explain why the question has a particular problem.
Please re-write the rationale in complete sentences to help users understand gaps in the question. 
Please keep your response between 20 and 50 words. 
Please start your response with "This question" followed by your rationale.

---

Follow the following format.

Question: The question to evaluate. The input will be a JSON with the following structure: { "cell_type": "question", "response_format": "open" or "closed", "description": string, "main_text": string, "response_categories": empty list for open questions or list of JSONs with an "id" and "text" field for closed questions }

Problem: The problem with the question. The input will be a string.

Input Rationale: The rationale for the problem. The input will be a string.

Reasoning: Let's think step by step in order to ${produce the new_ratio